3. سوالات تکلیف

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datetime import datetime
import os
from pathlib import Path
import sys


sys.path.append(str(Path.cwd().parent / 'src'))

from data_utils import get_moons_data, get_mnist_data

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_train, X_test, y_train, y_test, train_loader, test_loader = get_mnist_data(batch_size=64)

In [5]:

class MNIST_CNN(nn.Module):
    def __init__(self, kernel_size1=3):
        super(MNIST_CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=kernel_size1, stride=1, padding='same')
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2)  # 28x28 -> 14x14

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=1, padding='same')
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2)  # 14x14 -> 7x7

        self.fc = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


In [6]:

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [13]:
def train_and_evaluate(model, train_loader, test_loader, epochs=5, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.to(device)

    train_acc_history = []
    test_acc_history = []

    for epoch in range(epochs):
        model.train()
        correct_train = 0
        total_train = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, predicted = torch.max(outputs, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        train_acc = 100 * correct_train / total_train

        model.eval()
        correct_test = 0
        total_test = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total_test += labels.size(0)
                correct_test += (predicted == labels).sum().item()

        test_acc = 100 * correct_test / total_test
        train_acc_history.append(train_acc)
        test_acc_history.append(test_acc)
        if (epoch+1)%5==0:print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")
    return train_acc_history, test_acc_history


In [ ]:
kernel_sizes = [1, 3, 5, 7]
results = []

for ks in kernel_sizes:
    print(f"\n===== آزمایش با اندازه کرنل = {ks} =====")
    model = MNIST_CNN(kernel_size1=ks)
    num_params = count_parameters(model)
    print(f"تعداد پارامترهای مدل: {num_params:,}")

    train_acc_hist, test_acc_hist = train_and_evaluate(model, train_loader, test_loader, epochs=20, lr=0.001)

    final_test_acc = test_acc_hist[-1]
    results.append({
        'kernel_size': ks,
        'params': num_params,
        'final_test_acc': final_test_acc,
        'train_history': train_acc_hist,
        'test_history': test_acc_hist,
        'model': model
    })
    print(f"دقت نهایی آزمون برای کرنل {ks}: {final_test_acc:.2f}%")



===== آزمایش با اندازه کرنل = 1 =====
تعداد پارامترهای مدل: 20,362
Epoch 5/20 | Train Acc: 98.49% | Test Acc: 98.00%


In [ ]:

# 7. نمایش جدول نتایج
print("\n===== جدول نتایج =====")
df_results = pd.DataFrame([(r['kernel_size'], r['params'], r['final_test_acc']) for r in results],
                          columns=['اندازه کرنل', 'تعداد پارامترها', 'دقت آزمون (%)'])
print(df_results.to_string(index=False))

# ذخیره جدول به صورت CSV
df_results.to_csv('results_kernel_size.csv', index=False)
print("\nجدول نتایج در فایل 'results_kernel_size.csv' ذخیره شد.")

# 8. انتخاب بهترین مدل بر اساس دقت آزمون
best_result = max(results, key=lambda x: x['final_test_acc'])
best_ks = best_result['kernel_size']
best_model = best_result['model']
print(f"\nبهترین اندازه کرنل: {best_ks} با دقت {best_result['final_test_acc']:.2f}%")


In [ ]:

# 9. ترسیم فیلترهای یادگرفته شده لایه اول (وزن‌های کانولوشن)
def plot_filters(model, layer_name='conv1', save_path='filters_best_model.png'):
    # استخراج وزن‌های لایه مورد نظر
    for name, param in model.named_parameters():
        if layer_name in name and 'weight' in name:
            weights = param.data.cpu().numpy()  # shape: (out_channels, in_channels, h, w)
            break
    else:
        print(f"لایه {layer_name} یافت نشد.")
        return

    # تعداد فیلترها = out_channels
    n_filters = weights.shape[0]
    # برای MNIST ورودی تک‌کاناله است پس هر فیلتر یک ماتریس 2D است
    # ترسیم در یک grid
    cols = 4
    rows = (n_filters + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*2, rows*2))
    axes = axes.flatten()
    for i in range(n_filters):
        # نرمال‌سازی فیلتر برای نمایش بهتر
        filt = weights[i, 0, :, :]
        filt_min, filt_max = filt.min(), filt.max()
        filt = (filt - filt_min) / (filt_max - filt_min + 1e-8)
        axes[i].imshow(filt, cmap='gray')
        axes[i].set_title(f'Filter {i+1}')
        axes[i].axis('off')
    for i in range(n_filters, len(axes)):
        axes[i].axis('off')
    plt.suptitle(f'Filters of first convolutional layer (kernel size = {best_ks})')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"تصویر فیلترها در '{save_path}' ذخیره شد.")

plot_filters(best_model, save_path='filters_best_model.png')

# 10. نمایش نقشه ویژگی (feature map) برای یک تصویر آزمون
def plot_feature_maps(model, test_loader, device, save_path='feature_maps.png'):
    # یک batch از داده آزمون بگیریم
    model.eval()
    images, labels = next(iter(test_loader))
    img = images[0:1].to(device)  # فقط یک تصویر
    label = labels[0].item()

    # گرفتن خروجی لایه اول قبل از pooling و ReLU (برای دقت بیشتر)
    # باید forward را مرحله به مرحله انجام دهیم
    with torch.no_grad():
        x = model.conv1(img)       # خروجی کانولوشن
        x = model.relu1(x)
        # x = model.pool1(x)       # اگر بعد از پولینگ می‌خواهید، خط زیر را فعال کنید
        feature_maps = x[0].cpu().numpy()  # shape: (16, H, W)

    # نمایش تصویر ورودی
    img_display = img[0, 0].cpu().numpy()
    plt.figure(figsize=(12, 8))
    plt.subplot(3, 6, 1)
    plt.imshow(img_display, cmap='gray')
    plt.title(f'Input Image (label={label})')
    plt.axis('off')

    # نمایش 15 نقشه ویژگی اول (از 16)
    n_maps = min(15, feature_maps.shape[0])
    for i in range(n_maps):
        fm = feature_maps[i, :, :]
        fm_min, fm_max = fm.min(), fm.max()
        fm = (fm - fm_min) / (fm_max - fm_min + 1e-8)
        plt.subplot(3, 6, i+2)
        plt.imshow(fm, cmap='gray')
        plt.title(f'FM {i+1}')
        plt.axis('off')
    plt.suptitle(f'Feature maps after first convolutional layer (kernel size = {best_ks})')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"نقشه‌های ویژگی در '{save_path}' ذخیره شد.")

plot_feature_maps(best_model, test_loader, device, save_path='feature_maps_best_model.png')